# 2. Welch PSD Class Energy and Sensor Response Vector Map

This notebook uses the active-window interval obtained in Notebook 1 and converts each event-sensor response into a `Class 1-5` spectral response map.

The main outputs are:

```text
(1) Raw active-window response map
    = 24 sensors x active-window time samples

(2) Welch PSD class energy map
    = 24 sensors x 5 classes

(3) Sensor response vector map
    = sqrt(PSD class energy x active-window duration)
    = 24 sensors x 5 classes
```

The `Dominant Class` is recorded only as an interpretive label indicating the class with the largest PSD energy for each sensor. The health score calculation in Notebook 3 uses the full `Class 1-5` sensor response vector, not a single dominant class.


## 2.1 Raw Active-Window Response Map

The event-level active window extracted in Notebook 1 is used to generate a `24 sensors x active-window time samples` response map.

This step verifies which time interval of the measured signal is used before PSD calculation.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'aquinas_common.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Project root containing utils/aquinas_common.py was not found.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from utils.aquinas_common import (
    AXIS_LABEL_FONTSIZE,
    BANDS,
    BAND_COLS,
    COLORBAR_LABEL_FONTSIZE,
    COLORBAR_TICK_FONTSIZE,
    EPS,
    FS,
    PLOT_TITLE_FONTSIZE,
    TABLE_DIR,
    TICK_LABEL_FONTSIZE,
    band_energy_and_ratio_from_psd,
    configure_notebook_style,
    read_event_hdf5,
    method_sensor_index,
    sensor_order_table,
    set_sensor_index_axis,
    welch_psd,
)
from utils.aquinas_raw_class_trends import (
    RAW_CLASS_SIGNAL_FEATURE_PATH,
    load_or_build_dominant_class_signal_metric_table,
)

configure_notebook_style(dpi=150, grid=False)

CLASS_LABELS = [f'Class {idx}' for idx in range(1, len(BAND_COLS) + 1)]
CLASS_ENERGY_COLS = [f'class_{idx}_energy' for idx in range(1, len(BAND_COLS) + 1)]

sensor_metadata = pd.read_csv(
    TABLE_DIR / 'sensor_spectral_reference_fingerprint_v2.csv',
    usecols=['sensor_id', 'deck', 'sensor_alias', 'quantity', 'span', 'side', 'location', 'axis'],
    low_memory=False,
)
ordered_metadata = sensor_order_table(sensor_metadata)
reference_order = ordered_metadata[ordered_metadata['deck'].astype(str).str.upper().eq('NEW')].copy().reset_index(drop=True)
base_sensor_alias_order = reference_order['sensor_alias'].drop_duplicates().tolist()

sensor_index_table = (
    reference_order[[
        'sensor_alias',
        'quantity',
        'span',
        'side',
        'location',
        'axis',
        'support_group',
    ]]
    .drop_duplicates('sensor_alias')
    .reset_index(drop=True)
)
sensor_index_table['sensor_index'] = sensor_index_table['sensor_alias'].map(method_sensor_index).astype(int)
sensor_index_table['comparison_group'] = sensor_index_table['quantity'].astype(str) + ' ' + sensor_index_table['side'].astype(str)
plot_group_order = {'STR DO': 0, 'STR UP': 1, 'ACC DO': 2, 'ACC UP': 3}
span_order = {'S1': 0, 'S2': 1}
str_location_order = {'INF': 0, 'SHE': 1, 'SUP': 2, 'INT': 3, 'MID': 4}
acc_location_order = {'INT': 0, 'MID': 1, 'INF': 2, 'SHE': 3, 'SUP': 4}
axis_order = {'': 0, 'Y': 1, 'Z': 2}
sensor_index_table['plot_row'] = sensor_index_table['comparison_group'].map(plot_group_order).fillna(99).astype(int)
sensor_index_table['span_rank'] = sensor_index_table['span'].map(span_order).fillna(99).astype(int)
sensor_index_table['location_rank'] = np.where(
    sensor_index_table['quantity'].eq('ACC'),
    sensor_index_table['location'].map(acc_location_order).fillna(99),
    sensor_index_table['location'].map(str_location_order).fillna(99),
).astype(int)
sensor_index_table['axis_rank'] = sensor_index_table['axis'].fillna('').map(axis_order).fillna(99).astype(int)
local_group_order = (
    sensor_index_table
    .sort_values(['plot_row', 'comparison_group'])['comparison_group']
    .drop_duplicates()
    .tolist()
)
LOCAL_GROUP_LABELS = {group: f'G{idx + 1}' for idx, group in enumerate(local_group_order)}
sensor_index_table['local_group_id'] = sensor_index_table['comparison_group'].map(LOCAL_GROUP_LABELS)
sensor_plot_table = sensor_index_table.sort_values([
    'plot_row',
    'span_rank',
    'location_rank',
    'axis_rank',
    'sensor_index',
]).reset_index(drop=True)
sensor_group_alias_order = sensor_plot_table['sensor_alias'].tolist()
sensor_group_index_labels = sensor_plot_table['sensor_index'].astype(int).tolist()

# Display and example maps use grouped order, while the tick labels keep the method-level sensor indices.
sensor_alias_order = sensor_group_alias_order

signal_features = load_or_build_dominant_class_signal_metric_table(max_workers=8)
signal_features = signal_features[signal_features['sensor_alias'].isin(sensor_alias_order)].copy()
signal_features['deck'] = signal_features['deck'].astype(str).str.upper()

TARGET_EVENT_PATH = (
    PROJECT_ROOT
    / 'EWSHM_dataset_preprocessed_event_level'
    / 'AQUINAS_SET2_2023_04'
    / 'OLD'
    / 'PART020'
    / 'AQUINAS_SET2_OLD_2023_04_PART020_EVENT028_ID001266.hdf5'
)

target_event_path = str(TARGET_EVENT_PATH)
example_event_candidates = (
    signal_features.loc[signal_features['path'].eq(target_event_path), ['path', 'set_name', 'deck', 'event_id', 'campaign_month', 'part_index']]
    .drop_duplicates()
)
if example_event_candidates.empty:
    raise RuntimeError(f'Target event was not found in signal_features: {target_event_path}')
example_event_meta = example_event_candidates.iloc[0]
example_path = str(example_event_meta['path'])

event_rows = (
    signal_features[signal_features['path'].eq(example_path)]
    .set_index('sensor_alias')
    .reindex(sensor_alias_order)
)

active_start = float(pd.to_numeric(event_rows['active_start_s'], errors='coerce').dropna().median())
active_end = float(pd.to_numeric(event_rows['active_end_s'], errors='coerce').dropna().median())
if not np.isfinite(active_start) or not np.isfinite(active_end) or active_end <= active_start:
    raise RuntimeError('Valid active-window bounds were not found for the selected event.')

values, mask, aliases, time_grid = read_event_hdf5(example_path)
alias_to_idx = {alias: idx for idx, alias in enumerate(aliases)}
active_mask = (time_grid >= active_start) & (time_grid <= active_end)
active_time = time_grid[active_mask] - active_start

raw_active_map = np.full((len(sensor_alias_order), int(active_mask.sum())), np.nan, dtype=float)
valid_row_mask = np.zeros(len(sensor_alias_order), dtype=bool)
for row_idx, alias in enumerate(sensor_alias_order):
    sensor_idx = alias_to_idx.get(alias)
    if sensor_idx is None:
        continue
    valid = mask[sensor_idx].astype(bool) & active_mask
    if valid.sum() < 32:
        continue
    x = values[sensor_idx, active_mask].astype(float)
    local_valid = valid[active_mask]
    x = x - np.nanmedian(x[local_valid])
    x[~local_valid] = np.nan
    raw_active_map[row_idx] = x
    valid_row_mask[row_idx] = True

raw_abs_limit = max(EPS, float(np.nanpercentile(np.abs(raw_active_map), 99)))

fig, ax = plt.subplots(1, 1, figsize=(15.8, 7.2), constrained_layout=False)
fig.subplots_adjust(left=0.08, right=0.985, bottom=0.16, top=0.86)
fig.suptitle('2.1 Raw Active-Window Response Map', fontsize=PLOT_TITLE_FONTSIZE, y=0.96)

cmap = plt.get_cmap('seismic').copy()
cmap.set_bad('#f2f2f2')
im = ax.imshow(
    np.ma.masked_invalid(raw_active_map),
    aspect='auto',
    interpolation='nearest',
    cmap=cmap,
    vmin=-raw_abs_limit,
    vmax=raw_abs_limit,
    extent=[active_time[0], active_time[-1], len(sensor_alias_order) + 0.5, 0.5],
)
ax.set_xlabel('Time in active window (s)', fontsize=AXIS_LABEL_FONTSIZE)
ax.set_ylabel('Method-level sensor index', fontsize=AXIS_LABEL_FONTSIZE)
ax.set_yticks(np.arange(1, len(sensor_alias_order) + 1))
ax.set_yticklabels(sensor_group_index_labels, fontsize=TICK_LABEL_FONTSIZE)
ax.tick_params(axis='x', labelsize=TICK_LABEL_FONTSIZE)
cbar = fig.colorbar(im, ax=ax, orientation='horizontal', fraction=0.055, pad=0.12)
cbar.set_label('Zero-centered raw response, symmetric display limit', fontsize=COLORBAR_LABEL_FONTSIZE)
cbar.ax.tick_params(labelsize=COLORBAR_TICK_FONTSIZE)
plt.show()

print('Selected event')
print(f"  deck = {example_event_meta['deck']}")
print(f"  set = {example_event_meta['set_name']}")
print(f"  part = {int(example_event_meta['part_index'])}")
print(f"  event_id = {int(example_event_meta['event_id'])}")
print(f'  active window = {active_start:.2f} to {active_end:.2f} s')
print(f'  raw active-window map shape = {raw_active_map.shape[0]} sensors x {raw_active_map.shape[1]} samples')

## 2.2 Welch PSD Class Energy Map and Sensor Response Vector Map

Welch PSD is calculated for each sensor signal in the active window, and the PSD energy is integrated from `Class 1` to `Class 5`.

```text
active-window signal -> Welch PSD -> Class 1-5 PSD energy
```

Each event is first represented as:

```text
24 sensors x 5 classes PSD energy map
```

The active-window duration is then included to construct the sensor response vector map used in Notebook 3.

```text
Class response = sqrt(PSD class energy x active-window duration)
```

Therefore, panel (a) shows the PSD class energy map, while panel (b) shows the sensor response vector map that also reflects the active-window duration.


In [ ]:
psd_class_energy_map = np.zeros((len(sensor_alias_order), len(CLASS_LABELS)), dtype=float)
dominant_class_values = np.zeros(len(sensor_alias_order), dtype=int)

for row_idx, alias in enumerate(sensor_alias_order):
    x = raw_active_map[row_idx]
    x = x[np.isfinite(x)]
    if x.size < 64:
        psd_class_energy_map[row_idx, :] = np.nan
        continue
    freq, psd = welch_psd(x, fs=FS)
    energies, _ = band_energy_and_ratio_from_psd(freq, psd, bands=BANDS)
    psd_class_energy_map[row_idx, :] = energies
    if np.nansum(energies) > EPS:
        dominant_class_values[row_idx] = int(np.nanargmax(energies)) + 1

psd_class_energy_map[~valid_row_mask, :] = np.nan

active_duration_s = float(active_end - active_start)
class_response_map = np.sqrt(np.maximum(psd_class_energy_map, 0.0) * active_duration_s)
class_response_map[~valid_row_mask, :] = np.nan

energy_vmax = max(EPS, float(np.nanpercentile(psd_class_energy_map, 99)))
response_vmax = max(EPS, float(np.nanpercentile(class_response_map, 99)))

fig, axes = plt.subplots(1, 2, figsize=(15.8, 7.2), constrained_layout=False)
fig.subplots_adjust(left=0.08, right=0.985, bottom=0.18, top=0.84, wspace=0.18)
fig.suptitle('2.2 PSD Class Energy and Sensor Response Vector Map', fontsize=PLOT_TITLE_FONTSIZE, y=0.96)

cmap = plt.get_cmap('jet').copy()
cmap.set_bad('#f2f2f2')
plot_items = [
    (axes[0], psd_class_energy_map, '(a) Welch PSD Class energy map', 'Welch PSD Class energy', energy_vmax),
    (axes[1], class_response_map, '(b) Sensor response vector map', 'sqrt(PSD Class energy x active-window duration)', response_vmax),
]
for panel_idx, (ax, matrix, title, cbar_label, vmax) in enumerate(plot_items):
    im = ax.imshow(np.ma.masked_invalid(matrix), aspect='auto', interpolation='nearest', cmap=cmap, vmin=0.0, vmax=vmax)
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    set_sensor_index_axis(
        ax,
        sensor_alias_order,
        show_y=(panel_idx == 0),
        xlabel='Frequency class',
        band_ticklabels=CLASS_LABELS,
        xtick_fontsize=TICK_LABEL_FONTSIZE,
        ytick_fontsize=TICK_LABEL_FONTSIZE,
        xlabel_fontsize=AXIS_LABEL_FONTSIZE,
    )
    if panel_idx == 0:
        ax.set_yticks(np.arange(len(sensor_alias_order)))
        ax.set_yticklabels(sensor_group_index_labels, fontsize=TICK_LABEL_FONTSIZE)
        ax.set_ylabel('Method-level sensor index', fontsize=AXIS_LABEL_FONTSIZE)
    cbar = fig.colorbar(im, ax=ax, orientation='horizontal', fraction=0.055, pad=0.14)
    cbar.set_label(cbar_label, fontsize=COLORBAR_LABEL_FONTSIZE)
    cbar.ax.tick_params(labelsize=COLORBAR_TICK_FONTSIZE)
plt.show()

sensor_class_summary = pd.DataFrame({
    'sensor_index': sensor_group_index_labels,
    'sensor_alias': sensor_alias_order,
    'dominant_class': [f'Class {idx}' if 1 <= idx <= len(CLASS_LABELS) else 'missing' for idx in dominant_class_values],
    'dominant_psd_energy': [
        psd_class_energy_map[row_idx, class_id - 1] if 1 <= class_id <= len(CLASS_LABELS) else np.nan
        for row_idx, class_id in enumerate(dominant_class_values)
    ],
    'dominant_class_response': [
        class_response_map[row_idx, class_id - 1] if 1 <= class_id <= len(CLASS_LABELS) else np.nan
        for row_idx, class_id in enumerate(dominant_class_values)
    ],
})
display(sensor_class_summary)


## 2.3 Output Passed to Notebook 3

The key output passed to Notebook 3 is the `Class 1-5 sensor response vector` for each event and sensor.

This vector is constructed by incorporating the active-window duration into the PSD class energy.

```text
Class response = sqrt(PSD class energy x active-window duration)
```

This value is not the true vehicle input energy. It is a cumulative response-energy proxy defined from the measured structural response.


In [ ]:
handoff_table = sensor_class_summary.copy()
for class_idx, col in enumerate(CLASS_ENERGY_COLS, start=1):
    handoff_table[f'Class {class_idx} PSD energy'] = psd_class_energy_map[:, class_idx - 1]
    handoff_table[f'Class {class_idx} response'] = class_response_map[:, class_idx - 1]

handoff_table['active_duration_s'] = active_duration_s
handoff_table = handoff_table[[
    'sensor_index',
    'sensor_alias',
    'dominant_class',
    'dominant_psd_energy',
    'dominant_class_response',
    'active_duration_s',
    *[f'Class {idx} PSD energy' for idx in range(1, len(CLASS_LABELS) + 1)],
    *[f'Class {idx} response' for idx in range(1, len(CLASS_LABELS) + 1)],
]]

# display(handoff_table)

# print('Notebook 3 input')
# print('  unit of calculation = event x sensor')
# print(f'  PSD Class energy map shape = {psd_class_energy_map.shape[0]} sensors x {psd_class_energy_map.shape[1]} Classes')
# print(f'  sensor response vector map shape = {class_response_map.shape[0]} sensors x {class_response_map.shape[1]} Classes')
# print(f'  active-window duration = {active_duration_s:.3f} s')
# print('  dominant Class is kept as an interpretation label; Class 1-5 response values are used in Notebook 3.')


## 2.4 Summary

The main result of this notebook is:

```text
Raw active-window signal
-> Welch PSD
-> Class 1-5 PSD energy map
-> Sensor response vector map = sqrt(PSD class energy x active-window duration)
```

Notebook 3 uses the `Class 1-5 sensor response vector` to construct the sensor similarity matrix and compute sensor health scores by combining same-group consistency and opposite-span consistency.
